# An Explainable and Confidence-Calibrated Deep Learning Framework for Reliable Brain Tumor Classification from MRI

This notebook implements the complete four-class brain-tumor MRI pipeline:

**Custom CNN + ResNet50 + EfficientNet-B0 + DenseNet121 -> validation-based model selection -> temperature scaling -> uncertainty/abstention -> Grad-CAM**

> Research/educational use only; not for clinical diagnosis.

**Split protocol:** images from the original Kaggle `Training` and `Testing` folders are pooled and stratified into 70% train, 15% validation, and 15% test. This does not preserve the official Kaggle test partition; disclose this in the paper.


## 1. Install dependencies

Run this in Google Colab with a GPU runtime if available.


In [ ]:
!pip -q install kagglehub streamlit scikit-learn pandas matplotlib pillow
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Download the MRI dataset

In [ ]:
import kagglehub
dataset_root = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print(dataset_root)


## 3. Import the project code

When using the ZIP in Colab, upload and extract the project first, then `cd` into it. If this notebook is already inside the extracted project folder, the imports work directly.


In [ ]:
from src.core import *
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import json

seed_everything(42)
cfg = ProjectConfig(
    batch_size=32,
    head_epochs=3,
    fine_tune_epochs=5,
    target_coverage=0.90,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, val_loader, test_loader, class_names, split_info = build_loaders(dataset_root, cfg)
print("Device:", device)
print("Classes:", class_names)
print("Split summary:", json.dumps(split_info, indent=2))
print("Train batches:", len(train_loader), "Val batches:", len(val_loader), "Test batches:", len(test_loader))


## 4. Train all four architectures and select using validation only

The best architecture is selected using **validation macro F1**. Test-set results are added only after the selection is frozen. For a quick pipeline check, set `head_epochs=1` and `fine_tune_epochs=1`; for a final paper run, report the exact configuration.


In [ ]:
architectures = ["custom_cnn", "resnet50", "efficientnet_b0", "densenet121"]
trained = {}
rows = []

for arch in architectures:
    print("\nTraining:", arch)
    model, history, best_f1, best_epoch = fit_model(
        arch, len(class_names),
        (train_loader, val_loader, test_loader),
        cfg, device
    )
    trained[arch] = model
    val_logits, val_y = collect_logits(model, val_loader, device)
    val_probs = torch.softmax(val_logits, dim=1).numpy()
    val_metrics = compute_metrics(val_y.numpy(), val_probs)
    rows.append({
        "architecture": arch,
        "best_epoch": best_epoch,
        "best_val_f1_macro": best_f1,
        **{f"val_{key}": value for key, value in val_metrics.items()},
    })
    print("Validation metrics:", val_metrics)

comparison = pd.DataFrame(rows).sort_values("val_f1_macro", ascending=False)
best_arch = str(comparison.iloc[0]["architecture"])
print("Architecture selected by validation macro-F1:", best_arch)

# Selection is now frozen. Use the held-out test set only for final comparison/reporting.
for arch in architectures:
    test_logits_arch, test_y_arch = collect_logits(trained[arch], test_loader, device)
    test_probs_arch = torch.softmax(test_logits_arch, dim=1).numpy()
    test_metrics = compute_metrics(test_y_arch.numpy(), test_probs_arch)
    for key, value in test_metrics.items():
        comparison.loc[comparison["architecture"] == arch, f"test_{key}"] = value

comparison = comparison.sort_values("val_f1_macro", ascending=False)
comparison


## 5. Calibrate the validation-selected model

Temperature scaling is fitted on validation logits only. The test set is used only to measure the frozen model/calibration procedure.


In [ ]:
best_model = trained[best_arch]
print("Best model:", best_arch)

val_logits, val_y = collect_logits(best_model, val_loader, device)
test_logits, test_y = collect_logits(best_model, test_loader, device)

raw_val_probs = torch.softmax(val_logits, dim=1).numpy()
raw_test_probs = torch.softmax(test_logits, dim=1).numpy()

temperature = fit_temperature(val_logits, val_y)
cal_val_probs = softmax_with_temperature(val_logits, temperature).numpy()
cal_test_probs = softmax_with_temperature(test_logits, temperature).numpy()

print("Temperature:", temperature)
print("Test ECE before:", expected_calibration_error(raw_test_probs, test_y.numpy()))
print("Test ECE after :", expected_calibration_error(cal_test_probs, test_y.numpy()))
print("Test Brier before:", multiclass_brier(raw_test_probs, test_y.numpy()))
print("Test Brier after :", multiclass_brier(cal_test_probs, test_y.numpy()))


## 6. Validation-derived uncertainty threshold

The threshold is **not chosen arbitrarily**. It is learned from validation confidence to target a specified coverage (90% by default), then frozen and applied to the test set.


In [ ]:
threshold = choose_confidence_threshold(cal_val_probs, cfg.target_coverage)
print("Confidence threshold:", threshold)
print("Selective test metrics:", selective_metrics(cal_test_probs, test_y.numpy(), threshold))


## 7. Final classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
pred = cal_test_probs.argmax(axis=1)
print(classification_report(test_y.numpy(), pred, target_names=class_names, digits=4))
print("Confusion matrix:\n", confusion_matrix(test_y.numpy(), pred))


## 8. Grad-CAM explanation for one test MRI

In [ ]:
import matplotlib.pyplot as plt

x, y = next(iter(test_loader))
sample = x[0:1].to(device)
target_layer = get_gradcam_target_layer(best_model, best_arch)
cam_engine = GradCAM(best_model, target_layer)
cam, logits = cam_engine(sample)
cam_engine.close()

pred_idx = int(logits.argmax(dim=1).item())
img = x[0].permute(1,2,0).numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = np.clip(img * std + mean, 0, 1)

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.imshow(cam, alpha=0.4)
plt.axis("off")
plt.title(f"True: {class_names[int(y[0])]} | Pred: {class_names[pred_idx]}")
plt.show()


## 9. Export the final model bundle

In [ ]:
Path("outputs").mkdir(exist_ok=True)
with open("outputs/split_summary.json", "w") as f:
    json.dump(split_info, f, indent=2)
export_bundle(
    "outputs/best_model_bundle.pth",
    best_arch, best_model, class_names, cfg,
    temperature, threshold
)
print("Saved outputs/best_model_bundle.pth and outputs/split_summary.json")


## 10. Recommended final-paper result set

Report:

- Accuracy
- Macro Precision
- Macro Recall / Sensitivity
- Macro F1
- One-vs-Rest Macro AUC
- ECE before and after calibration
- Brier score before and after calibration
- Validation-derived confidence threshold
- Test coverage and selective accuracy
- Confusion matrix
- ROC curves
- Grad-CAM examples, including failures

For a publication claim about generalization, use a genuinely independent external dataset with no overlap with the composite Kaggle dataset.
